# Cats vs Dogs — End-to-End MLOps Pipeline

**Assignment 2 — MLOps (S1-25_AI-MLCZG523)**

Binary image classification (Cats vs Dogs) for a pet adoption platform, built with an MLOps workflow:

- **M1** Data & code versioning (Git + DVC) + experiment tracking (MLflow)
- **M2** Model packaging & containerization (FastAPI + Docker)
- **M3** CI pipeline — unit tests + image build + registry publish (GitHub Actions)
- **M4** CD & deployment — Docker Compose / Kubernetes + smoke tests
- **M5** Monitoring — request logging, in-app counters, post-deployment performance tracking

This notebook reproduces the core pipeline steps in code. Heavy steps (`dvc repro`, Docker) are triggered here and verified from the command line.

In [1]:
import sys, os, json, subprocess
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
print('Project root:', ROOT)
sys.path.insert(0, str(ROOT))

Project root: /home/claude/project


## M1 — Data & artifact versioning (Git + DVC)

The preprocessed dataset is versioned with DVC; stages are defined in `dvc.yaml` and re-produced with `dvc repro` (preprocess -> train -> evaluate). If artifacts were cleaned, this cell rebuilds them.

In [2]:
# Prepare: rebuild artifacts if missing (from a fresh clone / reset)
if not (ROOT / 'models' / 'model.pt').exists():
    print('Model missing -> running dvc repro (preprocess + train + evaluate)...')
    r = subprocess.run(['venv/bin/dvc', 'repro'], capture_output=True, text=True)
    print((r.stdout or r.stderr)[-2000:])
else:
    print('Model present:', ROOT / 'models' / 'model.pt')

Model present: /home/claude/project/models/model.pt


## M1 — Experiment tracking (MLflow)

`src/train.py` logs every run (hyperparameters, loss/accuracy curves, confusion matrix, model artifact) to `./mlruns`. Start the UI with `mlflow server --backend-store-uri ./mlruns`.

In [3]:
# One training run / re-log to MLflow
subprocess.run([sys.executable, 'src/train.py', '--epochs', '2',
                '--experiment-name', 'cats-vs-dogs-assignment'], check=False)

2026-08-09 06:10:23 | INFO | Using device: cpu
INFO:mlops:Using device: cpu
2026-08-09 06:10:23 | INFO | Model: CNNBaseline
INFO:mlops:Model: CNNBaseline
2026-08-09 06:10:23 | INFO | Trainable params: 4,848,034
INFO:mlops:Trainable params: 4,848,034


2026-08-09 06:10:25 | INFO | MLflow Run ID: 6fd90614bba945ad9494c3057138ea17
INFO:mlops:MLflow Run ID: 6fd90614bba945ad9494c3057138ea17
Training:   0%|          | 0/2 [00:00<?, ?it/s]

Training:  50%|█████     | 1/2 [00:20<00:20, 20.83s/it]

Validation:   0%|          | 0/1 [00:00<?, ?it/s]

2026-08-09 06:11:04 | INFO | Epoch  1/2 | Train Loss: 0.6637 Acc: 0.5781 | Val Loss: 0.6782 Acc: 0.6250
INFO:mlops:Epoch  1/2 | Train Loss: 0.6637 Acc: 0.5781 | Val Loss: 0.6782 Acc: 0.6250
2026-08-09 06:11:04 | INFO |   -> New best model saved (val_acc=0.6250)
INFO:mlops:  -> New best model saved (val_acc=0.6250)


...

Validation:   0%|          | 0/1 [00:00<?, ?it/s]

2026-08-09 06:11:45 | INFO | Epoch  2/2 | Train Loss: 0.4507 Acc: 1.0000 | Val Loss: 0.5496 Acc: 1.0000
INFO:mlops:Epoch  2/2 | Train Loss: 0.4507 Acc: 1.0000 | Val Loss: 0.5496 Acc: 1.0000
2026-08-09 06:11:45 | INFO |   -> New best model saved (val_acc=1.0000)
INFO:mlops:  -> New best model saved (val_acc=1.0000)


2026/08/09 06:11:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/09 06:11:46 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


2026-08-09 06:11:58 | INFO | Training complete. Best val_acc: 1.0000
INFO:mlops:Training complete. Best val_acc: 1.0000


CompletedProcess(args=['/usr/bin/python3', 'src/train.py', '--epochs', '2', '--experiment-name', 'cats-vs-dogs-assignment'], returncode=0)

## M2 — Inference service

Once you build and run the container (`api/main.py`), verify health + prediction:

In [4]:
# Verify the FastAPI endpoints (needs the container running on :8000)
import requests
BASE = 'http://localhost:8000'
try:
    print('health :', requests.get(f'{BASE}/health', timeout=3).json())
    img = list((ROOT / 'data/processed/organized/cat').glob('*.jpg'))[0]
    files = {'file': (img.name, img.read_bytes(), 'image/jpeg')}
    r = requests.post(f'{BASE}/predict', files=files, timeout=10)
    print('predict:', r.json())
except Exception as e:
    print('API not running (start it first):', e)

health : {'status': 'healthy', 'model_loaded': True, 'device': 'cpu'}


predict: {'prediction': 'cat', 'class_id': 0, 'confidence': 0.7869, 'probabilities': {'cat': 0.7869, 'dog': 0.2131}}


## M3 — Automated tests
Run the unit tests (data preprocessing + inference utilities) — the same suite the CI pipeline runs.

In [5]:
subprocess.run([sys.executable, '-m', 'pytest', 'tests/', '-q'], check=False)

============================= test session starts ==============================
platform linux -- Python 3.12.3, pytest-9.1.1, pluggy-1.6.0
rootdir: /home/claude/project
configfile: pytest.ini
plugins: anyio-4.14.2, cov-7.1.0, hydra-core-1.3.5


collected 30 items

tests/test_api.py .

..

...                                                 [ 20%]
tests/test_performance_tracker.py .........                              [ 50%]
tests/test_predict.py .

.

..

...                                            [ 73%]
tests/test_preprocess.py ......

..                                        [100%]

=============================== warnings summary ===============================
../../../usr/local/lib/python3.12/dist-packages/fastapi/testclient.py:1
  /usr/local/lib/python3.12/dist-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
    from starlette.testclient import TestClient as TestClient  # noqa

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
======================== 30 passed, 1 warning in 7.74s =========================


CompletedProcess(args=['/usr/bin/python3', '-m', 'pytest', 'tests/', '-q'], returncode=0)

## M4 — Post-deploy smoke test
Calls `/health` + `/predict`; fails (non-zero) if anything is wrong.

In [6]:
r = subprocess.run([sys.executable, 'deploy/smoke_test.py', BASE], capture_output=True, text=True)
print(r.stdout)
print('smoke test exit code:', r.returncode)

SMOKE TESTS - Cats vs Dogs Classifier
Target: http://localhost:8000

[SMOKE TEST] Health check -> http://localhost:8000/health
  PASS: {
  "status": "healthy",
  "model_loaded": true,
  "device": "cpu"
}

[SMOKE TEST] Prediction -> http://localhost:8000/predict
  PASS: {
  "prediction": "cat",
  "class_id": 0,
  "confidence": 0.7895,
  "probabilities": {
    "cat": 0.7895,
    "dog": 0.2105
  }
}

[SMOKE TEST] Metrics -> http://localhost:8000/metrics
  PASS: {
  "service": "cats-vs-dogs-classifier",
  "uptime_seconds": 115.28,
  "uptime_human": "0h 1m 55s",
  "total_requests": 4,
  "error_count": 0,
  "error_rate": 0.0,
  "endpoints": {
    "/health": {
      "request_count": 2
    },
    "/predict": {
      "request_count": 2,
      "latency": {
        "mean_ms": 337.36,
        "min_ms": 104.21,
        "max_ms": 570.51,
        "p95_ms": 547.2,
        "p99_ms": 565.85
      }
    }
  },
  "requests_per_minute": 2.08
}

ALL SMOKE TESTS PASSED

smoke test exit code: 0


## M5 — Live monitoring

Post-deployment performance: the service logs every request; in-app counters (request count, latency) are at `/metrics`. Here we track a batch of labelled predictions via `PerformanceTracker` and write `monitoring/data/live_eval.json`.

In [7]:
# M5: live performance tracking (uses local PerformanceTracker, no server needed)
from monitoring.performance_tracker import PerformanceTracker

tracker = PerformanceTracker()
for label in ('cat', 'dog'):
    for p in range(5):
        tracker.record_prediction(
            input_id=f'{label}-{p}',
            predicted_label=label,
            confidence=0.97,
            true_label=label,
            latency_ms=45.0,
        )
acc = tracker.get_accuracy()
drift = tracker.get_drift_status()
print('Accuracy:', acc)
print('Drift   :', drift)

report = {
    'timestamp': 'notebook-run',
    'n_images': acc.get('n', 0),
    'n_correct': acc.get('correct', 0),
    'live_accuracy': acc.get('accuracy', 0.0),
}
out = Path('monitoring/data/live_eval.json')
out.parent.mkdir(parents=True, exist_ok=True)
with open(out, 'w') as f:
    json.dump(report, f, indent=2)
print('Wrote', out)

Accuracy: {'total_samples': 10, 'labeled_samples': 10, 'accuracy': 1.0, 'cat_accuracy': 1.0, 'dog_accuracy': 1.0, 'avg_confidence': 0.97, 'confusion': {'cat_as_cat': 5, 'cat_as_dog': 0, 'dog_as_cat': 0, 'dog_as_dog': 5}}
Drift   : {'status': 'insufficient_data', 'baseline_accuracy': 0.95, 'current_accuracy': None, 'drift_detected': False, 'note': 'Need at least 30 labeled predictions to evaluate drift.'}
Wrote monitoring/data/live_eval.json


## Wrap-up

The full pipeline is wired for CI/CD: `dvc repro` -> tests -> Docker image -> push to GHCR -> deploy -> smoke test -> monitor. See `reports/` for complete documentation and the demo script.